
<div style="background: linear-gradient(135deg, #1a3a5c 0%, #2e75b6 100%); padding: 40px; border-radius: 12px; text-align: center; color: white; margin-bottom: 20px;">
<h1 style="font-size: 2.5em; margin-bottom: 10px;">🧹 Olist E-Commerce</h1>
<h2 style="font-weight: 300; font-size: 1.5em; margin-bottom: 20px;">Data Cleaning Notebook</h2>
<p style="font-size: 1.1em; opacity: 0.85;" >Step 1 of 6 · <b>integration</b> → EDA → Cleaning → Feature Engineering → Preprocessing → Modeling</p>
</div>

---

## What This Notebook Does

| Step | Action | Reason |
|---|---|---|
| **1** | Imports & load data | Start fresh from merged CSV |
| **2** | Snapshot before cleaning | Know what we started with |
| **3** | Filter to delivered orders | Non-delivered have no review/delivery date |
| **4** | Drop high-missing columns | 58–88% missing, not usable |
| **5** | Drop missing delivery dates | True data quality errors post-filter |
| **6** | Drop missing review scores | Can't model without the target |
| **7** | Impute product dimensions | Category-level median — smarter than global |
| **8** | Fill missing categories | Replace with 'unknown' |
| **9** | Drop remaining null rows | Payment, seller, price — very few |
| **10** | Cap price/freight outliers | Prevent extreme values from distorting models |
| **11** | Validate & save | Final check + export clean CSV |


#  Load Data
Load all 9 CSV files from the dataset folder into separate DataFrames.

In [16]:
import pandas as pd
 
DATA_PATH = "dataset"
 
customers    = pd.read_csv(f"{DATA_PATH}/olist_customers_dataset.csv")
orders       = pd.read_csv(f"{DATA_PATH}/olist_orders_dataset.csv")
order_items  = pd.read_csv(f"{DATA_PATH}/olist_order_items_dataset.csv")
payments     = pd.read_csv(f"{DATA_PATH}/olist_order_payments_dataset.csv")
reviews      = pd.read_csv(f"{DATA_PATH}/olist_order_reviews_dataset.csv")
products     = pd.read_csv(f"{DATA_PATH}/olist_products_dataset.csv")
sellers      = pd.read_csv(f"{DATA_PATH}/olist_sellers_dataset.csv")
translations = pd.read_csv(f"{DATA_PATH}/product_category_name_translation.csv")


# Check Shapes
Check the number of rows and columns in each DataFrame.


In [17]:
print(f"customers:    {list(customers.shape)}")
print(f"order_items:  {list(order_items.shape)}")
print(f"orders:       {list(orders.shape)}")
print(f"payments:     {list(payments.shape)}")
print(f"reviews:      {list(reviews.shape)}")
print(f"products:     {list(products.shape)}")
print(f"sellers:      {list(sellers.shape)}")
print(f"translations: {list(translations.shape)}")

customers:    [99441, 5]
order_items:  [112650, 7]
orders:       [99441, 8]
payments:     [103886, 5]
reviews:      [99224, 7]
products:     [32951, 9]
sellers:      [3095, 4]
translations: [71, 2]


# Check Columns
Check the column names of each DataFrame.

In [18]:
print(f"customers:    {list(customers.columns)}")
print(f"order_items:  {list(order_items.columns)}")
print(f"orders:       {list(orders.columns)}")
print(f"payments:     {list(payments.columns)}")
print(f"reviews:      {list(reviews.columns)}")
print(f"products:     {list(products.columns)}") 
print(f"sellers:      {list(sellers.columns)}")
print(f"translations: {list(translations.columns)}")

customers:    ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
order_items:  ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
orders:       ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
payments:     ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
reviews:      ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
products:     ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
sellers:      ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
t

# Check Duplicates
Check how many fully duplicated rows exist in each DataFrame.

In [19]:
print(f"customers:    {customers.duplicated().sum()}")
print(f"order_items:  {order_items.duplicated().sum()}")
print(f"orders:       {orders.duplicated().sum()}")
print(f"payments:     {payments.duplicated().sum()}")
print(f"reviews:      {reviews.duplicated().sum()}")
print(f"products:     {products.duplicated().sum()}") 
print(f"sellers:      {sellers.duplicated().sum()}")
print(f"translations: {translations.duplicated().sum()}")

customers:    0
order_items:  0
orders:       0
payments:     0
reviews:      0
products:     0
sellers:      0
translations: 0


#  Merge All Tables
Merge all 9 tables into one flat DataFrame using their shared keys:

In [21]:
df = (
    orders
    .merge(customers,    on="customer_id", how="left")
    .merge(order_items,  on="order_id",    how="left")
    .merge(payments,     on="order_id",    how="left")
    .merge(reviews,      on="order_id",    how="left")
    .merge(products,     on="product_id",  how="left")
    .merge(sellers,      on="seller_id",   how="left")
    .merge(translations, on="product_category_name", how="left")
)

print(df.shape)

(119143, 40)


In [ ]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,268.0,4.0,500.0,19.0,8.0,13.0,9350.0,maua,SP,housewares
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,268.0,4.0,500.0,19.0,8.0,13.0,9350.0,maua,SP,housewares
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,268.0,4.0,500.0,19.0,8.0,13.0,9350.0,maua,SP,housewares
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,178.0,1.0,400.0,19.0,13.0,19.0,31570.0,belo horizonte,SP,perfumery
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,232.0,1.0,420.0,24.0,19.0,21.0,14840.0,guariba,SP,auto


In [22]:
df.to_csv("olist_merged.csv", index=False)